In [1]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *
from qick.asm_v2 import AveragerProgramV2


/usr/local/share/pynq-venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)
/usr/local/share/pynq-venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)


In [2]:
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc


In [5]:


GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
#    -- pure numpy, identical to the v1 version, no API dependency
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0):
    n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- each step's buffer is a 4-tone composite
#    (ratio-weighted durations). All 4 tones shift together as the chirp
#    offset sweeps from CHIRP_OFFSET_START_HZ to CHIRP_OFFSET_STOP_HZ.
#    -- unchanged from v1 version
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ

NUM_STEPS = 50
AMPLITUDE = 0.5

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6 + 76.25e6, -147.82e6 + 76.25e6])
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])

BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3
STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
CYCLE_S = max_feasible_total_s / (NUM_STEPS + 1)

tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()
tone_durations_s = tone_fractions * CYCLE_S
print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held for "
      f"{STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

def build_multitone_buffer(chirp_offset_hz, phase0):
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur in zip(BASE_TONES_HZ, tone_durations_s):
        f = chirp_offset_hz + base_tone
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

chirp_offsets_hz = np.linspace(CHIRP_OFFSET_START_HZ, CHIRP_OFFSET_STOP_HZ, NUM_STEPS)
maxv = soccfg.get_maxv(GEN_CH)

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

samples_per_step = len(idata_list[0])
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0
assert min(piece_lens) >= 3 * samps_per_clk, (
    f"Smallest tone slice is only {min(piece_lens)} samples "
    f"({min(piece_lens)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3. "
    f"Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
)

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER -- unchanged
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM -- tProc v2 (QickProgramV2 / AveragerProgramV2)
#
#    Key differences from the v1 version:
#      * _initialize()/_body() replace initialize()/body() (note the leading _)
#      * No manual ch_page/sreg/mathi/loopnz register plumbing -- each
#        precomputed step buffer becomes a *named* pulse (add_envelope +
#        add_pulse), and _body() just calls self.pulse(..., t=<time>) once
#        per step in a normal Python for-loop. v2's assembler unrolls this
#        into real instructions itself, so the v1 "walk a register through
#        envelope memory across reps" trick isn't needed at all here.
#      * gain is normalized to -1.0..1.0 in v2 (not raw int16 like v1's
#        32767) -- since AMPLITUDE already scales the envelope samples
#        themselves, gain=1.0 means "no additional attenuation," matching
#        the intent of the v1 script's gain=32767.
#      * outsel="input" is preserved -- it has the same meaning in v2
#        (output = envelope table directly, no DDS multiplication).
#      * phrst=0 on every step pulse keeps the phase-coherent accumulator
#        running across pulses, matching the phase-continuous math already
#        built into build_multitone_buffer()/serrodyne_tone() above.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgramV2(AveragerProgramV2):
    def _initialize(self, cfg):
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            name = f"serr_{i}"
            self.add_envelope(ch=res_ch, name=name, idata=idata_step, qdata=qdata_step)
            self.add_pulse(
                ch=res_ch,
                name=name,
                style="arb",
                envelope=name,
                freq=0,
                phase=0,
                gain=cfg["gain"],
                outsel="input",
                mode="periodic",
                phrst=0,          # keep phase accumulator running across steps
            )

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])
        self.add_pulse(
            ch=res_ch,
            name="trap_wfm",
            style="arb",
            envelope="trap_wfm",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            outsel="input",
            mode="periodic",   # hardware repeats this forever once triggered
            phrst=0,
        )

    def _body(self, cfg):
        res_ch = cfg["res_ch"]
        step_hold_us = cfg["step_hold_us"]

        self.trigger(pins=[0], t=0)

        # Play each precomputed composite step in order, back-to-back.
        # No register hacking needed -- v2's assembler unrolls this loop
        # into real timed pulse instructions at compile time.
        t = 0.0
        for i in range(len(cfg["idata_list"])):
            self.pulse(ch=res_ch, name=f"serr_{i}", t=t)
            t += step_hold_us

        # Final trapping pulse: periodic, so the DAC keeps cycling the
        # 4 concatenated tones indefinitely regardless of what the tProc
        # does afterward -- including after the program ends.
        self.pulse(ch=res_ch, name="trap_wfm", t=t)

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "step_hold_us": STEP_HOLD_US,
    "gain": 1.0,   # v2 normalized full-scale; AMPLITUDE already scales the envelope itself
}

prog = RepeatedStepSerrodyneProgramV2(soccfg, reps=1, final_delay=0.5, cfg=config)
prog.run(soc)
print(f"Running on hardware -- {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 1: f_fabric=491.520 MHz, samps_per_clk=16, envelope sample rate=7.8643 GSPS
Envelope memory available: 65536 samples
Composite buffer cycle: 155.23 ns, held for 120.00 us per chirp step (6.000 ms total sweep)
  tone 0 (-76.2 MHz offset): ratio 0.337 -> requested 52.31 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 25.92 ns
  tone 2 (+46.7 MHz offset): ratio 0.288 -> requested 44.71 ns
  tone 3 (+71.6 MHz offset): ratio 0.208 -> requested 32.29 ns
Per-step composite buffer length: 1232 samples (tone slices: [416, 208, 352, 256], smallest = 13.0 fabric cycles)
Total envelope samples (sweep): 61600 / 65536 available
Trap buffer: 1232 samples (tone slices: [416, 208, 352, 256])
Total envelope samples (sweep + trap): 62832 / 65536 available
Running on hardware -- 50 sweep steps x 120.00 us = 6.000 ms sweep, then trapping (4 tones, periodic, indefinitely until soc.reset_gens()).


In [ ]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from qick import *
from qick.asm_v2 import AveragerProgramV2   # v2 classes aren't in `from qick import *`

soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

from scipy import signal as scipy_signal


def time_axis(num_samples, sample_rate):
    """Return a time axis in seconds."""
    return np.arange(int(num_samples), dtype=float) / float(sample_rate)


def serrodyne(
    ratios,
    freqs_hz,
    total_seconds,
    *,
    amplitude,
    sample_rate,
    max_points=None,
    width=1.0,
    continuous_phase=False,
):
    """Generate a piecewise serrodyne sawtooth waveform.
    Unchanged from the v1 version -- pure numpy, no QICK API dependency.
    """
    if len(ratios) != len(freqs_hz):
        raise ValueError("ratios and frequencies must have same length")
    if total_seconds <= 0:
        raise ValueError("total_seconds must be > 0")
    if not (0.0 <= width <= 1.0):
        raise ValueError("width must be in [0, 1]")

    sample_rate = float(sample_rate)
    n_samples = max(1, int(round(float(total_seconds) * sample_rate)))
    if max_points is not None:
        n_samples = min(n_samples, int(max_points))
    dt = 1.0 / sample_rate

    ratio_sum = sum(ratios)
    segment_lengths = [int(round(n_samples * (ratio / ratio_sum))) for ratio in ratios]
    delta = n_samples - sum(segment_lengths)
    index = 0
    while delta != 0 and index < len(segment_lengths) * 4:
        segment_index = index % len(segment_lengths)
        candidate = segment_lengths[segment_index] + (1 if delta > 0 else -1)
        if candidate >= 0:
            segment_lengths[segment_index] = candidate
            delta = n_samples - sum(segment_lengths)
        index += 1

    x = time_axis(n_samples, sample_rate)
    y = np.zeros(n_samples, dtype=float)
    start = 0
    phase_offset = 0.0
    two_pi = 2 * np.pi

    for length, frequency in zip(segment_lengths, freqs_hz):
        end = start + length
        if length > 0:
            t = time_axis(length, sample_rate)
            if frequency == 0:
                segment = np.zeros(length)
                if continuous_phase:
                    phase_offset = phase_offset % two_pi
            else:
                phase = (two_pi * float(frequency) * t) + (phase_offset if continuous_phase else 0.0)
                segment = float(amplitude) * (scipy_signal.sawtooth(phase, width=float(width)))
                if continuous_phase:
                    phase_offset = (two_pi * float(frequency) * (length * dt) + phase_offset) % two_pi
            y[start:end] = segment
        start = end

    return x, y, n_samples

# Generate a serrodyne waveform sized for this generator -- unchanged
gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6  # envelope sample rate, in Hz

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")

ratios = [0.337, 0.167, 0.288, 0.508]
freqs_hz = [76.25e6, 0, -122.92e6 + 76.25e6, -147.82e6 + 76.25e6]
requested_total_seconds = 1.0e-6

x, y, n_samples = serrodyne(
    ratios,
    freqs_hz,
    requested_total_seconds,
    amplitude=0.5,
    sample_rate=ENV_SR,
    continuous_phase=True,
)

pad = (-len(y)) % samps_per_clk
if pad:
    y = np.concatenate([y, np.zeros(pad)])

actual_total_seconds = len(y) / ENV_SR

print(f"Requested serrodyne period: {requested_total_seconds * 1e6:.6f} us")
print(f"Active samples: {n_samples}, padded to {len(y)} (multiple of {samps_per_clk})")
print(f"Actual serrodyne period: {actual_total_seconds * 1e6:.6f} us")

maxv = soccfg.get_maxv(GEN_CH)
idata = np.round(y * maxv).astype(np.int16)
qdata = np.zeros_like(idata)

fig, ax = plt.subplots(figsize=(11, 3))
t_ns = np.arange(len(idata)) / ENV_SR * 1e9
ax.plot(t_ns, idata)
ax.set_title("Serrodyne envelope (I data) to be loaded into the generator")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("DAC code")
ax.grid(True, alpha=0.3)

plt.savefig("/home/xilinx/jupyter_notebooks/qick/qick_demos/pictures/serrodyne_sending.pdf", bbox_inches="tight")
plt.close()

# -----------------------------------------------------------------------------
# PROGRAM -- tProc v2 (QickProgramV2 / AveragerProgramV2)
#
#   * initialize()/body() -> _initialize()/_body() (leading underscore)
#   * declare_gen / declare_readout / add_envelope are unchanged -- they're
#     defined in the shared qick_asm.py base, not v1- or v2-specific.
#   * set_pulse_registers(...) -> add_pulse(...) (register once, by name)
#     then self.pulse(ch=..., name=..., t=...) inside _body() to play it.
#   * v1's measure() shortcut (trigger+pulse+wait in one call) does NOT
#     exist in v2 -- call trigger() and pulse() directly instead.
#   * All timing is in microseconds in v2 (no us2cycles()/synci() needed --
#     the old adc_trig_offset of 100 clock ticks becomes a plain us value
#     below; you'll likely want to tune it empirically the same way you
#     would have tuned the clock-tick value, since the exact clock-tick ->
#     us conversion depends on the tProc fabric frequency).
#   * gain is normalized -1.0..1.0 in v2, not raw int16 -- gain=32767 (v1,
#     full scale) becomes gain=1.0 here, since AMPLITUDE already scales
#     the envelope samples themselves.
#   * relax_delay is handled by the `final_delay` constructor argument on
#     AveragerProgramV2, not by a manual syncdelay/us2cycles call at the
#     end of body().
# -----------------------------------------------------------------------------
class SerrodyneProgramV2(AveragerProgramV2):
    def _initialize(self, cfg):
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for ch in cfg["ro_chs"]:
            # This readout channel is "dynamic" (tProc-configured): declare_readout()
            # can't take freq/gen_ch here at all -- those are set separately below via
            # add_readoutconfig()/send_readoutconfig(), same pattern as the official
            # test01_basic_multipulse.ipynb v2 example.
            # Also note: length is in *microseconds* for v2 (it was raw samples in v1) --
            # readout_length_us below replaces the old "800 clock ticks" value.
            self.declare_readout(ch=ch, length=cfg["readout_length_us"])
            self.add_readoutconfig(ch=ch, name="ro_cfg", freq=0, gen_ch=res_ch)
            self.send_readoutconfig(ch=ch, name="ro_cfg", t=0)

        self.add_envelope(ch=res_ch, name="serrodyne",
                           idata=cfg["idata"], qdata=cfg["qdata"])

        self.add_pulse(
            ch=res_ch,
            name="serrodyne",
            style="arb",
            envelope="serrodyne",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            outsel="input",
            mode="periodic",
        )

    def _body(self, cfg):
        res_ch = cfg["res_ch"]
        self.trigger(ros=cfg["ro_chs"], pins=[0], t=cfg["adc_trig_offset_us"])
        self.pulse(ch=res_ch, name="serrodyne", t=0)

config = {
    "res_ch": GEN_CH,
    "ro_chs": [RO_CH],

    "readout_length_us": 1.5,    # [us] -- v2 uses microseconds here (v1 used raw samples/clock
                                  # ticks: 800). 1.5us comfortably covers the 1us serrodyne pulse
                                  # plus margin; tune if the capture window looks too tight/loose.
    "adc_trig_offset_us": 0.1,   # [us] -- was 100 clock ticks in v1; TUNE THIS empirically,
                                  # watching the captured trace, same as you would have tuned
                                  # the clock-tick value originally

    "idata": idata,
    "qdata": qdata,
    "gain": 1.0,                 # v2 normalized full-scale; AMPLITUDE already scales the envelope
}

prog = SerrodyneProgramV2(soccfg, reps=1, final_delay=1.0, cfg=config)
iq_list = prog.acquire_decimated(soc, progress=True)

# -----------------------------------------------------------------------------
# FIX: tProc v2's acquire_decimated() returns each readout's trace as shape
# (length, 2) -- i.e. one [I, Q] pair per row -- NOT (2, length) like v1 did.
# So `iq[0]`/`iq[1]` (the v1 pattern) no longer means "whole I trace" /
# "whole Q trace"; it means "sample 0's [I, Q] pair" / "sample 1's [I, Q]
# pair". That's what was causing "Number of samples: 2" -- iq[0] really was
# just a 2-element array. Use iq[:, 0] / iq[:, 1] instead.
# -----------------------------------------------------------------------------

plt.figure(figsize=(11, 3))
fs_iq = soccfg["readouts"][RO_CH]["f_output"] * 1e6
for ii, iq in enumerate(iq_list):
    t_us = np.arange(iq.shape[0]) / fs_iq * 1e6
    plt.plot(t_us, iq[:, 0], label="I value, ADC %d" % config["ro_chs"][ii])
    plt.plot(t_us, iq[:, 1], label="Q value, ADC %d" % config["ro_chs"][ii])

plt.ylabel("a.u.")
plt.xlabel("Time (µs)")
plt.title("Captured serrodyne pulse (loopback)")
plt.legend()
plt.show()
plt.savefig(
    "/home/xilinx/jupyter_notebooks/qick/qick_demos/pictures/serrodyne_recieved.pdf",
    bbox_inches="tight"
)
plt.savefig(
    "/home/xilinx/jupyter_notebooks/qick/qick_demos/pictures/serrodyne_recieved.png",
    bbox_inches="tight", dpi = 300
)
plt.close()

# --- everything below is pure numpy/plotting analysis, unchanged from v1 ---
# --- except for the I/Q indexing fix noted above ---

iq = iq_list[0]
I = np.asarray(iq[:, 0], dtype=float)
Q = np.asarray(iq[:, 1], dtype=float)
z = I + 1j * Q
N = len(z)

fs_iq = soccfg["readouts"][RO_CH]["f_output"] * 1e6

spectrum = np.fft.fftshift(np.abs(np.fft.fft(z))) / N
freqs_mhz = (np.fft.fftshift(np.fft.fftfreq(N, d=1 / fs_iq))) / 1e6

print(f"Number of samples: {N}")
print(f"IQ sample rate: {fs_iq / 1e6:.2f} MHz")
print(f"FFT resolution: {fs_iq / N / 1e6:.3f} MHz")
print(f"Frequency range: ±{fs_iq / 2 / 1e6:.2f} MHz")

plt.figure(figsize=(11, 3))
plt.plot(freqs_mhz, spectrum)
plt.xlabel("Frequency (MHz)")
plt.ylabel("Magnitude")
plt.title("Serrodyne spectrum — ADC loopback")
plt.grid(True, alpha=0.3)
plt.xlim(-150, 150)
plt.show()

signal = np.asarray(idata)
t_ns = np.asarray(t_ns)
N = len(signal)
dt = (t_ns[1] - t_ns[0]) * 1e-9
fs = 1 / dt

spectrum = np.fft.fftshift(np.abs(np.fft.fft(signal))) / N
freqs_mhz = (np.fft.fftshift(np.fft.fftfreq(N, d=dt))) / 1e6

print(f"Number of samples: {N}")
print(f"Sample spacing: {dt * 1e9:.3f} ns")
print(f"Sample rate: {fs / 1e6:.2f} MHz")
print(f"FFT resolution: {fs / N / 1e6:.3f} MHz")
print(f"Frequency range: ±{fs / 2 / 1e6:.2f} MHz")

plt.figure(figsize=(11, 3))
plt.plot(freqs_mhz, spectrum)
plt.xlabel("Frequency (MHz)")
plt.ylabel("Magnitude")
plt.title("Ideal serrodyne spectrum")
plt.grid(True, alpha=0.3)
plt.xlim(-150, 150)
plt.show()

def plot_eom_spectrum(phase_codes, sample_rate, title="Expected EOM spectrum"):
    data = np.asarray(phase_codes, dtype=float)

    span = float(np.max(data) - np.min(data))
    if span == 0.0:
        raise ValueError("Cannot plot spectrum for a constant waveform")

    phase = 2 * np.pi * (data - np.min(data)) / span
    field = np.exp(1j * phase)
    N = len(field)

    spectrum = np.fft.fftshift(np.abs(np.fft.fft(field))) / N
    freqs_mhz = (np.fft.fftshift(np.fft.fftfreq(N, d=1 / sample_rate))) / 1e6

    print(f"Number of samples: {N}")
    print(f"Sample rate: {sample_rate / 1e9:.3f} GS/s")
    print(f"FFT resolution: {sample_rate / N / 1e6:.3f} MHz")
    print(f"Frequency range: \u00b1{sample_rate / 2 / 1e6:.2f} MHz")

    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(freqs_mhz, spectrum)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-350, 350)
    plt.show()

    return ax

# FIX: index the I channel with [:, 0], not [0], for the same reason as above.
I_captured = np.asarray(iq_list[0][:, 0], dtype=float)
threshold = 0.2 * np.max(np.abs(I_captured))
active_idx = np.where(np.abs(I_captured) > threshold)[0]
if len(active_idx) == 0:
    raise RuntimeError("No samples above threshold -- check loopback cabling / gain / threshold.")
i0, i1 = active_idx[0], active_idx[-1] + 1
I_active = I_captured[i0:i1]

print(f"Active window: samples [{i0}:{i1}] out of {len(I_captured)} captured "
      f"({len(I_active)} active, {len(I_captured) - len(I_active)} dead-time samples trimmed)")

fs_readout = soccfg["readouts"][RO_CH]["f_output"] * 1e6

plot_eom_spectrum(
    I_active,
    fs_readout,
    title="Expected serrodyne EOM spectrum (active samples only)"
)
plt.savefig("/home/xilinx/jupyter_notebooks/qick/qick_demos/pictures/serrodyne_fft_shifts.pdf", bbox_inches="tight")
plt.savefig("/home/xilinx/jupyter_notebooks/qick/qick_demos/pictures/serrodyne_fft_shifts.png", bbox_inches="tight", dpi = 300)
plt.close()

In [4]:
soc.reset_gens()